# Silver Layer — Clean & Enrich Healthcare Data
Validate admissions, deduplicate records, derive length of stay buckets and readmission risk flags.

In [ ]:
from pyspark.sql.functions import (
    col, when, lit, to_timestamp, date_format, hour,
    datediff, row_number, current_timestamp, trim, upper
)
from pyspark.sql.window import Window

In [ ]:
# Pass-through staff catalog — already clean dimension
df_staff = spark.read.format('delta').table('bronze_staff_catalog')
df_staff = df_staff.withColumn('silver_timestamp', current_timestamp())
df_staff.write.mode('overwrite').format('delta').saveAsTable('silver_staff_catalog')
print(f'Silver staff catalog: {df_staff.count()} rows')

In [ ]:
# Clean patient admissions
df_adm = spark.read.format('delta').table('bronze_patient_admissions')

# Deduplicate on admission_id
w = Window.partitionBy('admission_id').orderBy(col('ingestion_timestamp').desc())
df_adm = (
    df_adm
    .withColumn('_rn', row_number().over(w))
    .filter(col('_rn') == 1)
    .drop('_rn')
)

df_adm = (
    df_adm
    .withColumn('admission_date',  to_timestamp('admission_date'))
    .withColumn('discharge_date',  to_timestamp('discharge_date'))
    .withColumn('length_of_stay_days', col('length_of_stay_days').cast('int'))
    .withColumn('is_readmission',  col('is_readmission').cast('boolean'))
    .filter(col('admission_date').isNotNull())
    .filter(col('length_of_stay_days') >= 0)
    # Derive LOS bucket
    .withColumn('los_bucket',
        when(col('length_of_stay_days') == 0, 'Same Day')
        .when(col('length_of_stay_days') <= 2, '1-2 Days')
        .when(col('length_of_stay_days') <= 7, '3-7 Days')
        .when(col('length_of_stay_days') <= 14, '8-14 Days')
        .otherwise('15+ Days'))
    # Admission day-of-week and hour
    .withColumn('admission_date_only', date_format('admission_date', 'yyyy-MM-dd'))
    .withColumn('admission_hour', hour('admission_date'))
    .withColumn('admission_shift',
        when(hour('admission_date') < 8, 'Night')
        .when(hour('admission_date') < 16, 'Day')
        .otherwise('Evening'))
    # High-risk flag: ICU, readmission, or LOS > 7
    .withColumn('high_risk_flag',
        (col('department') == 'ICU') |
        (col('is_readmission') == True) |
        (col('length_of_stay_days') > 7))
    .withColumn('silver_timestamp', current_timestamp())
)

df_adm.write.mode('overwrite').format('delta').saveAsTable('silver_patient_admissions')
print(f'Silver admissions: {df_adm.count()} rows')

In [ ]:
# Clean clinical records (vitals)
df_clin = spark.read.format('delta').table('bronze_clinical_records')

w2 = Window.partitionBy('record_id').orderBy(col('ingestion_timestamp').desc())
df_clin = (
    df_clin
    .withColumn('_rn', row_number().over(w2))
    .filter(col('_rn') == 1)
    .drop('_rn')
)

df_clin = (
    df_clin
    .withColumn('recorded_at', to_timestamp('recorded_at'))
    .withColumn('value', col('value').cast('double'))
    .filter(col('recorded_at').isNotNull())
    .filter(col('value').isNotNull())
    # Flag abnormal vitals
    .withColumn('is_abnormal',
        when((col('vital_type') == 'blood_pressure_systolic') & ((col('value') < 90) | (col('value') > 140)), True)
        .when((col('vital_type') == 'heart_rate') & ((col('value') < 60) | (col('value') > 100)), True)
        .when((col('vital_type') == 'temperature_c') & ((col('value') < 36.1) | (col('value') > 37.8)), True)
        .when((col('vital_type') == 'o2_saturation_pct') & (col('value') < 95), True)
        .otherwise(False))
    .withColumn('recorded_date', date_format('recorded_at', 'yyyy-MM-dd'))
    .withColumn('silver_timestamp', current_timestamp())
)

df_clin.write.mode('overwrite').format('delta').saveAsTable('silver_clinical_records')
print(f'Silver clinical records: {df_clin.count()} rows')